In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

# Load the file from Downloads (update path as needed)
df = pd.read_csv("/Users/kimballweeks/Downloads/cleaned_data.csv")
# Rename columns to make them formula-safe
df = df.rename(columns={
    "pct_highschool_or_more (1990)": "pct_hs_1990",
    "Pop 2023": "pop_2023"
})

# Convert church_density_1890 to numeric (remove 'DNE' entries)
df['church_density_1890'] = pd.to_numeric(df['church_density_1890'], errors='coerce')

# Drop rows where church_density_1890 is missing
df = df.dropna(subset=['church_density_1890'])

# Create log-transformed variables
df['log_income_2023'] = np.log(df['income_2023'])
df['log_pop_2023'] = np.log(df['pop_2023'])

# Standardize 1989 income and 1990 education
scaler = StandardScaler()
df[['income_1989_scaled', 'pct_hs_1990_scaled']] = scaler.fit_transform(
    df[['income_1989_real_2023', 'pct_hs_1990']]
)

# Run regression with robust standard errors
model = smf.ols(
    formula='log_income_2023 ~ church_density_1890 + income_1989_scaled + pct_hs_1990_scaled + log_pop_2023 + C(State)',
    data=df
).fit(cov_type='HC3')

# Display regression results
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:        log_income_2023   R-squared:                       0.780
Model:                            OLS   Adj. R-squared:                  0.776
Method:                 Least Squares   F-statistic:                     157.6
Date:                Fri, 18 Jul 2025   Prob (F-statistic):               0.00
Time:                        23:01:32   Log-Likelihood:                 1957.3
No. Observations:                2659   AIC:                            -3813.
Df Residuals:                    2608   BIC:                            -3512.
Df Model:                          50                                         
Covariance Type:                  HC3                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              11.0221    